# Building Your First Real AI Agent: An Invoice Approval Agent
### Python + LangGraph + Groq

This notebook builds an agent that reads a messy invoice, checks it against spending policy, uses an
LLM only for genuine judgment calls, and either auto-approves it or escalates it.

It mirrors how AP automation tools (Ramp, Brex, SAP Concur) handle expenses at scale.

**Tested against:** langgraph 1.2.12, langchain-groq 1.1.3, langchain-core 1.6.6 (October 2026).


## Step 1 - Install the stack

`langgraph` is the graph/agent framework. `langchain-groq` talks to Groq through a standard interface.
`langchain-core` comes along for the ride (message types, etc.).

In [ ]:
%pip install -q langgraph langchain-groq langchain-core

## Step 2 - Handle the API key safely

Get a free key at https://console.groq.com. We never hardcode it; `getpass` keeps it out of the
notebook output and out of version control. For a real project use a `.env` file with `python-dotenv`.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")


## Step 3 - Pick the model and sanity-check the connection

Groq retires models often, so we keep the name in one constant, `MODEL`. (`llama-3.3-70b-versatile`
was retired mid-build, so we moved to `openai/gpt-oss-120b`.) `temperature=0` makes the agent
deterministic - the same invoice gets the same decision, which matters when code approves money.

In [ ]:
from langchain_groq import ChatGroq

MODEL = "openai/gpt-oss-120b"   # swap here if Groq retires it; see console.groq.com for the live list
llm = ChatGroq(model=MODEL, temperature=0)

resp = llm.invoke("Reply with exactly this sentence: Groq is wired up.")
print(resp.content)


## Step 4 - Define the agent's state (the shared notepad)

State is the memory that travels through every node. Declare its shape once as a `TypedDict`. Each node
reads from it and returns a small dict of only the fields it changed; LangGraph merges that update back
in. You do not return the whole state, and you do not mutate it in place.

In [ ]:
from typing import TypedDict, Optional

class InvoiceState(TypedDict):
    # input
    raw_text: str

    # filled by the extract node (LLM)
    vendor: Optional[str]
    amount: Optional[float]
    currency: Optional[str]
    category: Optional[str]
    description: Optional[str]   # short plain-English summary; gives the judge real context
    date: Optional[str]

    # filled by the policy_check node (code)
    route: Optional[str]          # "auto_ok" | "needs_judgment" | "hard_fail"
    policy_notes: Optional[list]  # why the router decided what it did

    # filled by judge / finalize
    decision: Optional[str]       # "APPROVED" | "ESCALATED"
    reason: Optional[str]         # human-readable explanation


## Step 5 - The extract node: messy text in, clean fields out

Our first LLM node. Instead of parsing text ourselves, we hand the model a Pydantic schema and get back
a typed object. The field `description`s are prompt engineering - the model reads them.
`method="json_schema", strict=True` turns on constrained decoding so gpt-oss cannot return anything that
doesn't match the schema. Safe defaults mean the deterministic code downstream can trust the shape.

In [ ]:
from pydantic import BaseModel, Field

class ExtractedInvoice(BaseModel):
    vendor: str = Field(description="Company or person being paid. Empty string if not found.")
    amount: float = Field(description="Total amount owed as a number, e.g. 1234.56. Use 0 if not found.")
    currency: str = Field(description="3-letter code like USD, EUR, INR. Empty string if not stated.")
    category: str = Field(description="One of: office_supplies, travel, meals, software, hardware, services, other.")
    description: str = Field(description="One short line, in plain English, describing what was actually purchased.")
    date: str = Field(description="Invoice date as YYYY-MM-DD if determinable, else empty string.")

extractor = llm.with_structured_output(ExtractedInvoice, method="json_schema", strict=True)

def extract_node(state: InvoiceState) -> dict:
    """LLM job: pull structured fields out of raw invoice text."""
    result = extractor.invoke(
        "Extract the fields from this invoice. If a field is missing, use the "
        "empty-string / zero default described in the schema. Do not invent values.\n\n"
        f"INVOICE:\n{state['raw_text']}"
    )
    return {
        "vendor": result.vendor,
        "amount": result.amount,
        "currency": result.currency,
        "category": result.category,
        "description": result.description,
        "date": result.date,
    }


## Step 6 - Test the node in isolation (before any graph exists)

A node is just a function, so call it directly with a hand-made state. Always get a node working alone
before wiring it into the graph.

In [ ]:
messy_invoice = """
INVOICE  #4471
From: Blue Ridge Office Supplies Co.
Billed to: Acme Corp
Date: March 3, 2026
------------------------------------
20x reams printer paper .......... $180.00
2x toner cartridges .............. $240.00
------------------------------------
TOTAL DUE: $420.00 (USD)
Thanks for your business!
"""

print(extract_node({"raw_text": messy_invoice}))


## Step 7 - The policy_check node: pure rules, no LLM

The deterministic heart of the agent. No model. It applies spending policy with plain Python and writes
a `route` label plus `policy_notes` (an audit trail). Policy lives as data in a dict. Three outcomes:
`hard_fail` (a human must look), `auto_ok` (approve without the LLM), `needs_judgment` (gray area for
the judge).

In [ ]:
AUTO_APPROVE_LIMITS = {
    "office_supplies": 500,
    "meals": 150,
    "software": 1000,
    "hardware": 1000,
    "travel": 1500,
    "services": 1000,
    "other": 0,       # "other" is never auto-approved; always at least a judgment call
}
HARD_CEILING = 5000   # above this, a human looks no matter the category

def policy_check_node(state: InvoiceState) -> dict:
    """CODE job: apply spending rules, set a route label and an audit trail."""
    notes = []
    amount = state["amount"] or 0
    vendor = (state["vendor"] or "").strip()
    category = state["category"] or "other"

    if not vendor:
        notes.append("Missing vendor/payee.")
    if amount <= 0:
        notes.append("Amount missing or not parseable.")
    if notes:
        return {"route": "hard_fail", "policy_notes": notes}

    if amount > HARD_CEILING:
        notes.append(f"Amount {amount} exceeds hard ceiling {HARD_CEILING}.")
        return {"route": "hard_fail", "policy_notes": notes}

    limit = AUTO_APPROVE_LIMITS.get(category, 0)
    if amount <= limit:
        notes.append(f"{amount} within {category} auto-approve limit {limit}.")
        return {"route": "auto_ok", "policy_notes": notes}

    notes.append(f"{amount} over {category} limit {limit} but under ceiling {HARD_CEILING}.")
    return {"route": "needs_judgment", "policy_notes": notes}


## Step 8 - The router: reads the label, names the next node

The conditional edge, kept separate from the node. It does no work - it just reads the `route` the
policy node wrote and returns the next node's name. Node = work, edge = routing.

In [ ]:
def route_after_policy(state: InvoiceState) -> str:
    """Conditional edge: return the next node's name based on state."""
    return state["route"]   # "hard_fail" | "auto_ok" | "needs_judgment"


## Step 9 - Test the policy logic in isolation

Prove the rules are right with hand-made states before wiring anything.

In [ ]:
cases = {
    "small office supplies (expect auto_ok)":
        {"vendor": "Blue Ridge", "amount": 420.0, "currency": "USD", "category": "office_supplies", "date": "2026-03-03"},
    "pricey client dinner (expect needs_judgment)":
        {"vendor": "The Grill", "amount": 380.0, "currency": "USD", "category": "meals", "date": "2026-03-04"},
    "huge invoice (expect hard_fail: ceiling)":
        {"vendor": "BigCo", "amount": 90000.0, "currency": "USD", "category": "services", "date": "2026-03-05"},
    "no vendor (expect hard_fail: validity)":
        {"vendor": "", "amount": 50.0, "currency": "USD", "category": "office_supplies", "date": ""},
}

for label, fields in cases.items():
    out = policy_check_node(fields)
    print(f"{label:48} -> {out['route']:14} | {out['policy_notes']}")


## Step 10 - The judge node: the one real gray-area call

Our second and last LLM node, firing only on `needs_judgment`. We give it the fields, the `policy_notes`,
and the raw invoice text, so it reasons about what was actually bought rather than a vague label. It
returns a structured decision (`APPROVED` / `ESCALATED`) plus a reason. When in doubt it escalates.

In [ ]:
from typing import Literal

class JudgeDecision(BaseModel):
    decision: Literal["APPROVED", "ESCALATED"] = Field(
        description="APPROVED only if clearly reasonable business spend; ESCALATED if a human should review.")
    reason: str = Field(description="One or two plain-language sentences explaining the decision.")

judge = llm.with_structured_output(JudgeDecision, method="json_schema", strict=True)

def judge_node(state: InvoiceState) -> dict:
    """LLM job: the single gray-area judgment call."""
    prompt = (
        "You approve business expenses. Decide APPROVED or ESCALATED.\n"
        "Approve only if the spend is clearly reasonable and legitimate for a business.\n"
        "Escalate if it is unusual, ambiguous, possibly personal, or needs a manager's eye.\n\n"
        f"vendor: {state['vendor']}\n"
        f"amount: {state['amount']} {state['currency']}\n"
        f"category: {state['category']}\n"
        f"description: {state['description']}\n"
        f"date: {state['date']}\n"
        f"why this reached you: {state['policy_notes']}\n\n"
        f"ORIGINAL INVOICE TEXT:\n{state['raw_text']}"
    )
    result = judge.invoke(prompt)
    return {"decision": result.decision, "reason": result.reason}


## Step 11 - The finalize node: one consistent output

All paths converge here. For `auto_ok` / `hard_fail` the decision is determined by policy, so finalize
writes it from the notes. For `needs_judgment` the judge already wrote `decision` and `reason`, so
finalize returns an empty update (a real LangGraph pattern) and leaves them untouched.

In [ ]:
def finalize_node(state: InvoiceState) -> dict:
    """CODE job: produce one consistent final decision + explanation."""
    route = state["route"]
    if route == "auto_ok":
        return {"decision": "APPROVED",
                "reason": "Auto-approved by policy: " + "; ".join(state["policy_notes"])}
    if route == "hard_fail":
        return {"decision": "ESCALATED",
                "reason": "Escalated to a human: " + "; ".join(state["policy_notes"])}
    # needs_judgment: the judge already set decision + reason, so keep them.
    return {}


## Step 12 - Wire the graph (the nodes become an agent)

Now we assemble the map. `add_node` registers each function. `add_edge` draws the fixed arrows.
`add_conditional_edges` is the fork: source node, our `route_after_policy` function, and a dict mapping
each label it can return to the next node. `compile()` turns the map into a runnable agent. You build the
graph only now, after every node works on its own.

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(InvoiceState)
builder.add_node("extract", extract_node)
builder.add_node("policy_check", policy_check_node)
builder.add_node("judge", judge_node)
builder.add_node("finalize", finalize_node)

builder.add_edge(START, "extract")
builder.add_edge("extract", "policy_check")
builder.add_conditional_edges(
    "policy_check",
    route_after_policy,
    {"auto_ok": "finalize", "needs_judgment": "judge", "hard_fail": "finalize"},
)
builder.add_edge("judge", "finalize")
builder.add_edge("finalize", END)

agent = builder.compile()
print("Agent compiled.")


## Step 13 - See the graph you just built

`draw_mermaid()` prints Mermaid text (no installs) you can paste into https://mermaid.live. Seeing the
forks match the architecture you drew at the start is a good gut-check that the wiring is correct.

In [ ]:
print(agent.get_graph().draw_mermaid())


## Step 14 - Run the whole agent end to end

Three invoices, one per path. The office-supplies one auto-approves with no LLM judge; the client dinner
goes to the judge because it is over the meals limit but under the ceiling; the big software license is
escalated by policy alone. Same agent, three behaviors - exactly the design we drew at the start.

(We assign each result to a variable so Jupyter doesn't echo the full state dict under the printout.)

In [ ]:
def run_agent(invoice_text: str):
    result = agent.invoke({"raw_text": invoice_text})
    print("ROUTE   :", result["route"])
    print("DECISION:", result["decision"])
    print("REASON  :", result["reason"])
    print("-" * 60)
    return result

gray_invoice = """
INVOICE 2231
Vendor: The Capital Grille
Date: 2026-03-04
Business dinner, 6 attendees (client + team)
Total: $380.00 USD
"""

big_invoice = """
INVOICE 9001
Vendor: Enterprise Software Ltd
Annual platform license
Amount due: $42,000.00 USD
Date: 2026-03-01
"""

result_1 = run_agent(messy_invoice)   # auto_ok -> APPROVED
result_2 = run_agent(gray_invoice)    # needs_judgment -> judge decides
result_3 = run_agent(big_invoice)     # hard_fail -> ESCALATED
